# Englacial Temperature Profiles and Thermistor Metrics

This notebook produces interpolated 2D englacial temperature cross-sections and computes thermistor-derived metrics for all six study glaciers.

**Part 1 – Interpolated temperature profiles**
- GPR bedrock picks define the glacier geometry; borehole thermistors constrain the temperature field
- Radial Basis Function (RBF) interpolation generates a continuous 2D temperature field
- Firn cover grids are overlaid to contextualise the thermal structure
- The cold-temperate transition surface (CTS) is identified and plotted for each cross-section
- Publication figures: Alphubel (fig06), Chessjen (fig07), Hohsaas combined L1+L2 (fig08)

**Part 2 – Zero Annual Amplitude (ZAA) and thermistor metrics**
- Seasonal amplitude decay fitted per borehole to estimate ZAA depth
- Summary statistics across all six sites


---

## Part 1 — Interpolated Temperature Profiles

## Import necessary Libraries and Moudules

In [ ]:
from config import ICETEMP_ROOT, GPR_ROOT
%matplotlib inline
import sys
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, BoundaryNorm
import matplotlib.gridspec as gridspec
import matplotlib.image as mpimg
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable

mpl.rcParams['figure.dpi'] = 100

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import src.gpr_processing as gpr
import src.gpr_plotting as gprp
from src.gpr_plotting import *
from src.thermistor_processing import *
from src.thermistor_plotting import *
from src.plot_composer import *


## Preprocessing

### Set directions to thermistor data

In [ ]:
# set main gpr data dir
root_dir = GPR_ROOT + "/"

# set main icetemp data dir
gp_icetemp_dir = os.path.join(ICETEMP_ROOT, "thermistor_chains", "temperature_data") + "/"
TT_icetemp_dir = os.path.join(ICETEMP_ROOT, "NTC_tynitag", "temperature_data", "full_timeseries") + "/"

# set path to current depth file
depth_AH1G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah1g.csv")
depth_AH2G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah2g.csv")
depth_AH3G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah3g.csv")
depth_AH1TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah1tt.csv")
depth_AH2TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah2tt.csv")
depth_AH3TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_ah3tt.csv")

depth_CJ1G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj1g.csv")
depth_CJ2G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj2g.csv")
depth_CJ1TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj1tt.csv")
depth_CJ2TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj2tt.csv")
depth_CJ3TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj3tt.csv")
depth_CJ4TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_cj4tt.csv")

depth_HS1G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs1g.csv")
depth_HS2G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs2g.csv")
depth_HS3G = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs3g.csv")
depth_HS1TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs1tt.csv")
depth_HS2TT = os.path.join(project_root, "data", "borehole_settings", "thermistor_settings_hs2tt.csv")

# set chain data dir
A551FE_dir = gp_icetemp_dir + "2025/A551FE/raw/A551FE_20250916075641.csv" # AH1G
A55204_dir = gp_icetemp_dir + "2025/A55204/raw/A55204_20250916082002.csv" # AH2G
A55205_dir = gp_icetemp_dir + "2025/A55205/raw/A55205_20250916074654.csv" # AH3G
A551FD_dir = gp_icetemp_dir + "2025/A551FD/raw/A551FD_20250927153221.csv" # HS1G
A55203_dir = gp_icetemp_dir + "2025/A55203/raw/A55203_20250927151514.csv" # HS2G
A55200_dir = gp_icetemp_dir + "2025/A55200/raw/A55200_20250927151301.csv" # HS3G
A55201_dir = gp_icetemp_dir + "2025/A55201/raw/A55201_20250903090341.csv" # CJ1G
A55202_dir = gp_icetemp_dir + "2025/A55202/raw/A55202_20250903111312.csv" # CJ2G

# set tiny tag data dir
AH3TT_dir = TT_icetemp_dir + "AH3TT_20250806_20250916.csv"
CJ1TT_dir = TT_icetemp_dir + "CJ1TT_20240809_20250808_spliced.csv"
CJ2TT_dir = TT_icetemp_dir + "CJ2TT_20240809_20250808_spliced.csv"
CJ3TT_dir = TT_icetemp_dir + "/../2024_2025/new_naming/CJ3TT_20251215.csv"
CJ4TT_dir = TT_icetemp_dir + "/../2024_2025/new_naming/CJ4TT_20251215.csv"
HS1TT_dir = TT_icetemp_dir + "HS1TT_20240808_20250927_spliced.csv"
HS2TT_dir = TT_icetemp_dir + "HS2TT_20240808_20250927_spliced.csv"

# generate a thermistor data object
AH1G = ThermistorData(A551FE_dir, ",", depth_AH1G)
AH2G = ThermistorData(A55204_dir, ",", depth_AH2G)
AH3G = ThermistorData(A55205_dir, ",", depth_AH3G)
AH3TT = ThermistorData(AH3TT_dir, ",", depth_AH3TT)

CJ1G = ThermistorData(A55201_dir, ",", depth_CJ1G)
CJ2G = ThermistorData(A55202_dir, ",", depth_CJ2G)
CJ1TT = ThermistorData(CJ1TT_dir, ",", depth_CJ1TT)
CJ2TT = ThermistorData(CJ2TT_dir, ",", depth_CJ2TT)
CJ3TT = ThermistorData(CJ3TT_dir, ",", depth_CJ3TT)
CJ4TT = ThermistorData(CJ4TT_dir, ",", depth_CJ4TT)

HS1G = ThermistorData(A551FD_dir, ",", depth_HS1G)
HS2G = ThermistorData(A55203_dir, ",", depth_HS2G)
HS3G = ThermistorData(A55200_dir, ",", depth_HS3G)
HS1TT = ThermistorData(HS1TT_dir, ",", depth_HS1TT)
HS2TT = ThermistorData(HS2TT_dir, ",", depth_HS2TT)

# read chain temperature offsets from CSV file
offsets_path_G = os.path.join(project_root, "products", "figures", "thermistor_calibration", "corrected_chain_offsets.csv")
offsets_path_TT = os.path.join(ICETEMP_ROOT, "NTC_tynitag", "calibration_data", "all_logger_offsets.csv")
corrected_offsets_G = pd.read_csv(offsets_path_G, index_col='chain')
corrected_offsets_TT = pd.read_csv(offsets_path_TT)

In [ ]:
## Step 1. get coordinates for each borehole
bh_csv = os.path.join(project_root, "data", "borehole_settings", "thermistor_coordinates.csv")
borehole_coordinates, bh_missing = gpr.load_borehole_positions(bh_csv)

## Step 2. get temperature data for each borehole and thermistor

# Chessjen boreholes
CJ1G_data = CJ1G.get_chain_data_with_offsets(start_time='20250811', end_time='20250903', offsets=corrected_offsets_G.loc["A55201"], aggregate='mean')
CJ2G_data = CJ2G.get_chain_data_with_offsets(start_time='20250811', end_time='20250903', offsets=corrected_offsets_G.loc["A55202"], aggregate='mean')
CJ1TT_data = CJ1TT.get_ntc_data_with_offsets('7', corrected_offsets_TT, aggregate='all') # average over entire period
CJ2TT_data = CJ2TT.get_ntc_data_with_offsets('8', corrected_offsets_TT, aggregate='all') # average over entire period
CJ3TT_data = CJ3TT.get_ntc_data_with_offsets('14', corrected_offsets_TT, aggregate='all') # average over entire period
CJ4TT_data = CJ4TT.get_ntc_data_with_offsets('15', corrected_offsets_TT, aggregate='all') # average over entire period

# Alphubel boreholes
AH1G_data = AH1G.get_chain_data_with_offsets(start_time='20250810', end_time='20250916', offsets=corrected_offsets_G.loc["A551FE"], aggregate='mean')
AH2G_data = AH2G.get_chain_data_with_offsets(start_time='20250810', end_time='20250916', offsets=corrected_offsets_G.loc["A55204"], aggregate='mean')
AH3G_data = AH3G.get_chain_data_with_offsets(start_time='20250810', end_time='20250916', offsets=corrected_offsets_G.loc["A55205"], aggregate='mean')
AH3TT_data = AH3TT.get_ntc_data_with_offsets('13', corrected_offsets_TT, aggregate='all') # average over entire period

# Hohsaas boreholes
HS1G_data = HS1G.get_chain_data_with_offsets(start_time='20250815', end_time='20250927', offsets=corrected_offsets_G.loc["A551FD"], aggregate='mean')
HS2G_data = HS2G.get_chain_data_with_offsets(start_time='20250815', end_time='20250927', offsets=corrected_offsets_G.loc["A55203"], aggregate='mean')
HS3G_data = HS3G.get_chain_data_with_offsets(start_time='20250815', end_time='20250927', offsets=corrected_offsets_G.loc["A55200"], aggregate='mean')
HS1TT_data = HS1TT.get_ntc_data_with_offsets('5', corrected_offsets_TT, aggregate='all') # average over entire period
HS2TT_data = HS2TT.get_ntc_data_with_offsets('6', corrected_offsets_TT, aggregate='all') # average over entire period

## step 3. get depths for each thermistor in the borehole

# Chessjen boreholes
CJ1G_depths = read_thermistor_depths(depth_CJ1G)
CJ2G_depths = read_thermistor_depths(depth_CJ2G)
CJ1TT_depths = read_thermistor_depths(depth_CJ1TT)
CJ2TT_depths = read_thermistor_depths(depth_CJ2TT)
CJ3TT_depths = read_thermistor_depths(depth_CJ3TT)
CJ4TT_depths = read_thermistor_depths(depth_CJ4TT)

# Alphubel boreholes
AH1G_depths = read_thermistor_depths(depth_AH1G)
AH2G_depths = read_thermistor_depths(depth_AH2G)
AH3G_depths = read_thermistor_depths(depth_AH3G)
AH3TT_depths = read_thermistor_depths(depth_AH3TT)

# Hohsaas boreholes
HS1G_depths = read_thermistor_depths(depth_HS1G)
HS2G_depths = read_thermistor_depths(depth_HS2G)
HS3G_depths = read_thermistor_depths(depth_HS3G)
HS1TT_depths = read_thermistor_depths(depth_HS1TT)
HS2TT_depths = read_thermistor_depths(depth_HS2TT)

### Round temperature data to 2 digits
Otherwise e.g. temperatures at -0.001 will be displayed as cold -> Soon needs to be improved to adjust for the melting point temperature

In [ ]:
# Round temperature-like columns in a DataFrame
def round_temp_columns(df: pd.DataFrame, decimals: int = 5):
    if not isinstance(df, pd.DataFrame):
        return df
    # GeoPrecision chain columns: '#1', '#2', ... (leave TIME, NO, HK-BAT:V untouched)
    chain_cols = [c for c in df.columns if re.fullmatch(r"#\d+", str(c))]
    # TinyTag probe columns
    tt_cols = [c for c in ["Black Probe Temperature", "White Probe Temperature"] if c in df.columns]
    cols = chain_cols + tt_cols
    if cols:
        df[cols] = df[cols].apply(pd.to_numeric, errors="coerce").round(decimals)
    return df

# Apply to all loaded datasets if present
datasets = [
    "CJ1G_data","CJ2G_data","AH1G_data","AH2G_data","AH3G_data",
    "HS1G_data","HS2G_data","HS3G_data",
    "CJ1TT_data","CJ2TT_data","CJ3TT_data","CJ4TT_data","HS1TT_data","HS2TT_data","AH3TT_data"
]
for name in datasets:
    df = globals().get(name)
    if isinstance(df, pd.DataFrame):
        round_temp_columns(df, 3)

### Get GPR bedrock picks and generate profiles

In [ ]:
# List GPR bedrock picks files for each glacier/profile
gpr_picks_dirs = [
    root_dir + "/20250515_Alphubel/bed picks_south/20060101_GPR_picks_south.txt", # Alphubel south
    root_dir + "/20240809_Chessjen/picks/20240809_chessjen_picks_updated.csv", # Chessjen
    root_dir + "/20240808_Hohsaas/picks/20240808_hohsaas_picks.csv", # Hohsaas (if available)
]

# Build a profile table (use the raw TXT files)
df_all_AH = gpr.load_points_from_txt(gpr_picks_dirs[0], epsg=2056, drop_duplicates=True, aggregate_duplicates='mean', return_type='df')

# Choose a profile id present in df_all['profile'] (e.g., 12)
prof_id_AH = 4
prof_AH = gpr.extract_profile_table(df_all_AH, prof_id_AH, order_method="pca")

# Build a profile table for Chessjen (use the raw CSV files)
df_all_cj = gpr.load_points_from_csv(gpr_picks_dirs[1], epsg=2056, source_epsg=32632, drop_duplicates=True, aggregate_duplicates='mean', return_type='df')

# Choose a profile id present in df_all_cj['profile'] (e.g., 47)
prof_id_cj = 47
prof_cj = gpr.extract_profile_table(df_all_cj, prof_id_cj, order_method="pca")

# Build a profile table for Hohsaas (use the raw CSV files)
df_all_hs = gpr.load_points_from_csv(gpr_picks_dirs[2], epsg=2056, source_epsg=32632, drop_duplicates=True, aggregate_duplicates='mean', return_type='df')

# Choose a profile id present in df_all_hs['profile'] (e.g., 40)
prof_hs1 = gpr.extract_profile_table(df_all_hs, 39, order_method="pca")
prof_hs2 = gpr.extract_profile_table(df_all_hs, 40, order_method="pca")

df_all_hs_drone = gpr.load_points_from_csv(root_dir + "20250928_Hohsaas_data_drone_gpr/centerline_picks/20250928_Hohsaas_drone_gpr_centerline_picks.csv", epsg=2056, source_epsg=2056, drop_duplicates=False, aggregate_duplicates='mean', return_type='df')

# alternative drone-based GPR profile for Hohsaas
prof_hs_dronegpr1 = gpr.extract_profile_table(df_all_hs_drone, 1, order_method="pca")
prof_hs_dronegpr2 = gpr.extract_profile_table(df_all_hs_drone, 2, order_method="pca")

## Plot Alphubel interpolated englacial temperature profile

In [ ]:
# Create the input dictionaries for Alphubel
thermistor_data_AH = {
    'AH1G': AH1G_data,
    'AH2G': AH2G_data,
    'AH3G': AH3G_data,
    'AH3TT': AH3TT_data
}

depth_data_AH = {
    'AH1G': AH1G_depths,
    'AH2G': AH2G_depths,
    'AH3G': AH3G_depths,
    'AH3TT': AH3TT_depths
}

# Extract temperature and depth dictionaries
temp_data_dict_AH, depth_dict_AH = create_temp_depth_dicts(thermistor_data_AH, depth_data_AH)

## Create custom vik colormap

In [ ]:
def truncate_colormap(cmap, minval=0.0, maxval=0.5, n=256):
    """Return a truncated version of a colormap (e.g., just the cold half)."""
    new_cmap = LinearSegmentedColormap.from_list(
        f'trunc({cmap.name},{minval:.2f},{maxval:.2f})',
        cmap(np.linspace(minval, maxval, n)))
    return new_cmap

# Example usage in your notebook
cold_vik = truncate_colormap(cmc.vik, minval=0.0, maxval=0.8)  # 0.0 is cold end, 1.0 is warm end

In [ ]:
fig, ax = gprp.plot_icetemp_profile(
    profile_df=prof_AH,
    borehole_coords_df=borehole_coordinates,
    temp_data_dict=temp_data_dict_AH,
    depth_dict=depth_dict_AH,
    flip=False,          # or 'auto'
    n_depth=200,
    n_elev=200,
    temp_step=0.25,
    plot_contours=True,
    break_threshold=50.0,  # optional (default 50.0)
    smooth_sigma=0.0,      # increase (e.g. 20) to smooth surface/bed & CTS mask
    show_cts=True,
    adjust_cts_for_pressure=True,
    cts_tol=0.1,
    bed_uncertainty=5.0,       # ±2 m whiskers along bed
    bed_unc_every=20.0,        # every 20 m
    bed_unc_color= "#E69F00",
    bh_marker_size=6.0,
    bh_line_lw=1.2,
    bed_unc_lw=1.2,
    bed_unc_capsize=4.0,
    panel_tag="L4",
    tag_bbox={"facecolor":"white","edgecolor":"red","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    # cmap=plt.cm.viridis,  # pass a Colormap object instead of a string to avoid TypeError
    cbar_min=-3.5,
    cbar_tick_step=0.5,
    export_txt_path=project_root + "/results/interpolated_icetemps/updated_runs/20250916_icetemps_AH_profile_4.txt",
    export_borehole_txt_path=project_root + "/results/interpolated_icetemps/updated_runs/20250916_icetemps_AH_boreholes.txt",
    export_cts_mask_path=project_root + "/results/interpolated_icetemps/updated_runs/20250916_icetemps_AH_cts_mask_4.txt",
    continuous_cmap=False,
    rbf_smooth=0.05,
)

## Plot Chessjen interpolated englacial temperature profile

In [ ]:
# Create the input dictionaries
thermistor_data_CJ = {
    'CJ1G': CJ1G_data,
    'CJ2G': CJ2G_data, 
    'CJ1TT': CJ1TT_data,
    'CJ2TT': CJ2TT_data,
    'CJ3TT': CJ3TT_data,
    'CJ4TT': CJ4TT_data
}

depth_data_CJ = {
    'CJ1G': CJ1G_depths,
    'CJ2G': CJ2G_depths,
    'CJ1TT': CJ1TT_depths,
    'CJ2TT': CJ2TT_depths,
    'CJ3TT': CJ3TT_depths,
    'CJ4TT': CJ4TT_depths
}

# Extract temperature and depth dictionaries
temp_data_dict_CJ, depth_dict_CJ = create_temp_depth_dicts(thermistor_data_CJ, depth_data_CJ)

In [ ]:
# plot Chessjen profile from borehole temperature data
fig, ax = gprp.plot_icetemp_profile(
    profile_df=prof_cj,
    borehole_coords_df=borehole_coordinates,
    temp_data_dict=temp_data_dict_CJ,
    depth_dict=depth_dict_CJ,
    flip=True,
    n_depth=100,
    n_elev=200,
    temp_step=0.25,
    plot_contours=True,
    break_threshold=10.0,
    smooth_sigma=50.0,
    show_cts=True,
    adjust_cts_for_pressure=True,
    cts_tol=0.1,
    bed_uncertainty=5.0,       # ±2 m whiskers along bed
    bed_unc_every=30.0,        # every 20 m
    bed_unc_color= "#E69F00",
    bh_marker_size=6.0,
    bh_line_lw=1.2,
    bed_unc_lw=1.2,
    bed_unc_capsize=4.0,
    panel_tag="L47",
    tag_bbox={"facecolor":"white","edgecolor":"red","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    cbar_min=-3.0,
    cbar_tick_step=0.5,
    export_txt_path=project_root + "/results/interpolated_icetemps/updated_runs/20251215_icetemps_CJ_profile_47.txt",
    export_borehole_txt_path=project_root + "/results/interpolated_icetemps/updated_runs/20251215_icetemps_CJ_boreholes.txt",
    export_cts_mask_path=project_root + "/results/interpolated_icetemps/updated_runs/20251215_icetemps_CJ_cts_mask_47.txt",
    continuous_cmap=False,
    # NEW: Hatching regions
    hatch_regions=[(270, 300)],  # Distance ranges to hatch
    # hatch_pattern='///',         # Pattern: '///', '\\\\', 'xxx', '|||', etc.
    # hatch_color='lightgrey',     # Hatch line color
    hatch_fill_color='white',
    hatch_linewidth=0.6,         # Thickness of hatch lines
    hatch_alpha=0.0,    
)

## Plot Hohsaas interpolated englacial temperature profile

2 Profiles: 
1. Line 39 (thermistors HS1G and HS2TT)
2. Line 40 (thermistors HS2G and HS1TT)

In [ ]:
# Create the input dictionaries for Hohsaas
thermistor_data_HS1 = {
    'HS1G': HS1G_data,
    'HS2TT': HS2TT_data
}

thermistor_data_HS2 = {
    'HS2G': HS2G_data,
    'HS1TT': HS1TT_data
}

depth_data_HS1 = {
    'HS1G': HS1G_depths,
    'HS2TT': HS2TT_depths
}

depth_data_HS2 = {
    'HS2G': HS2G_depths,
    'HS1TT': HS1TT_depths
}

# Extract temperature and depth dictionaries
temp_data_dict_HS1, depth_dict_HS1 = create_temp_depth_dicts(thermistor_data_HS1, depth_data_HS1)
temp_data_dict_HS2, depth_dict_HS2 = create_temp_depth_dicts(thermistor_data_HS2, depth_data_HS2)

### Interpolate drone based gpr centerlines

In [ ]:
# Create the input dictionaries for Hohsaas drone GPR profiles centerlines
thermistor_data_HS_drone_1 = {
    'HS1TT': HS1TT_data,
    'HS2TT': HS2TT_data,
    'HS2G': HS2G_data
}

thermistor_data_HS_drone_2 = {
    'HS1G': HS1G_data,
    'HS2G': HS2G_data,
    'HS3G': HS3G_data
}

depth_data_HS_drone_1 = {
    'HS1TT': HS1TT_depths,
    'HS2TT': HS2TT_depths,
    'HS2G': HS2G_depths
}

depth_data_HS_drone_2 = {
    'HS1G': HS1G_depths,
    'HS2G': HS2G_depths,
    'HS3G': HS3G_depths
}

# Extract temperature and depth dictionaries
temp_data_dict_HS_drone_1, depth_dict_HS_drone_1 = create_temp_depth_dicts(thermistor_data_HS_drone_1, depth_data_HS_drone_1)
temp_data_dict_HS_drone_2, depth_dict_HS_drone_2 = create_temp_depth_dicts(thermistor_data_HS_drone_2, depth_data_HS_drone_2)

In [ ]:
fig, ax = gprp.plot_icetemp_profile(
    profile_df=prof_hs_dronegpr1,
    borehole_coords_df=borehole_coordinates,
    temp_data_dict=temp_data_dict_HS_drone_1,
    depth_dict=depth_dict_HS_drone_1,
    flip=False,          # or 'auto'
    n_depth=100,
    n_elev=200,
    temp_step=0.2,
    plot_contours=True,
    break_threshold=50.0,  # optional (default 50.0)
    smooth_sigma=6.0,      # increase (e.g. 20) to smooth surface/bed & CTS mask
    show_cts=True,
    adjust_cts_for_pressure=True,
    cts_tol=0.05,
    bed_uncertainty=5.0,       # ±2 m whiskers along bed
    bed_unc_every=20.0,        # every 20 m
    bed_unc_color= "darkgrey",
    bh_marker_size=6.0,
    bh_line_lw=1.2,
    bed_unc_lw=1.2,
    bed_unc_capsize=4.0,
    panel_tag="L1",
    panel_tag_color="darkorange",
    tag_bbox={"facecolor":"white","edgecolor":"darkorange","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    # cmap=plt.cm.viridis,  # pass a Colormap object instead of a string to avoid TypeError
    cbar_min=-3.5,
    cbar_tick_step=0.5,
    export_txt_path=project_root + "/results/interpolated_icetemps/updated_runs/20250916_icetemps_HS_dronegpr1_profile.txt",
    export_borehole_txt_path=project_root + "/results/interpolated_icetemps/updated_runs/20250916_icetemps_HS_boreholes_dronegpr1.txt",
    export_cts_mask_path=project_root + "/results/interpolated_icetemps/updated_runs/20250916_icetemps_HS_dronegpr1_cts_mask.txt",
    continuous_cmap=False,
    hatch_regions=[(245, 400)],
    hatch_pattern='',              # Empty = no hatching
    hatch_fill_color='white',  # Solid fill
    hatch_alpha=0.8                # 50% transparent 
)

In [ ]:
fig, ax = gprp.plot_icetemp_profile(
    profile_df=prof_hs_dronegpr2,
    borehole_coords_df=borehole_coordinates,
    temp_data_dict=temp_data_dict_HS_drone_2,
    depth_dict=depth_dict_HS_drone_2,
    flip=False,          # or 'auto'
    n_depth=100,
    n_elev=200,
    temp_step=0.25,
    plot_contours=True,
    break_threshold=50.0,  # optional (default 50.0)
    smooth_sigma=6.0,      # increase (e.g. 20) to smooth surface/bed & CTS mask
    show_cts=True,
    adjust_cts_for_pressure=True,
    cts_tol=0.05,
    bed_uncertainty=5.0,       # ±2 m whiskers along bed
    bed_unc_every=20.0,        # every 20 m
    bed_unc_color= "grey",
    bh_marker_size=6.0,
    bh_line_lw=1.2,
    bed_unc_lw=1.2,
    bed_unc_capsize=4.0,
    panel_tag="L2",
    panel_tag_color="darkorange",
    tag_bbox={"facecolor":"white","edgecolor":"darkorange","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    # cmap=plt.cm.viridis,  # pass a Colormap object instead of a string to avoid TypeError
    cbar_min=-3.5,
    cbar_tick_step=0.5,
    export_txt_path=project_root + "/results/interpolated_icetemps/updated_runs/20250916_icetemps_HS_dronegpr2_profile.txt",
    export_borehole_txt_path=project_root + "/results/interpolated_icetemps/updated_runs/20250916_icetemps_HS_boreholes_dronegpr2.txt",
    export_cts_mask_path=project_root + "/results/interpolated_icetemps/updated_runs/20250916_icetemps_HS_dronegpr2_cts_mask.txt",
    continuous_cmap=False,
    rbf_smooth=0.14,
    # NEW: Hatching regions
    hatch_regions=[(260, 400)],
    hatch_pattern='',              # Empty = no hatching
    hatch_fill_color='white',  # Solid fill
    hatch_alpha=0.9                # 50% transparent 
)

### Plot Hohsaas profiles side by side

In [ ]:
labels_HS1 = ['HS1G','HS2TT']
labels_HS2 = ['HS2G','HS1TT']
label_colors_HS1 = build_profile_color_map(labels_HS1)
label_colors_HS2 = build_profile_color_map(labels_HS2)

fig, axs = gprp.plot_icetemp_profiles_side_by_side(
    profiles=[
        (prof_hs2, temp_data_dict_HS2, depth_dict_HS2),
        (prof_hs1, temp_data_dict_HS1, depth_dict_HS1),
    ],
    borehole_coords_df=borehole_coordinates,
    panel_tags=["L40","L39"],       # small corner labels
    flips=[False, False],
    n_depth=400, n_elev=600, temp_step=0.25,
    plot_contours=True, break_threshold=10.0, smooth_sigma=50.0,
    show_cts=True, adjust_cts_for_pressure=True, cts_tol=0.05,
    borehole_clip_buffer=[10.0, 10.0],
    label_colors_list=[label_colors_HS2, label_colors_HS1],
    figsize=(14.5,7.5),
    panel_gap=0.16,                 # extra whitespace so ticks don’t crowd
    bh_label_offset=3.0,
    y_top_margin=20.0,
    keep_true_slope=True,           # enforce comparable metric scale across panels
    show_all_y_ticks=True,
    bed_uncertainty=[5.0, 5.0],   # per-panel, constant ±2 m
    bed_unc_every=20.0,
    bed_unc_color= "#767676",
    tag_bbox={"facecolor":"white","edgecolor":"black","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    cbar_min=-3.5,
    cbar_tick_step=0.5,       # shared colorbar ticks
    bh_marker_size=10.0,
    bh_line_lw=2.0,
    bed_unc_lw=2.0,
    bed_unc_capsize=4.0,
)

## Plot combined interpolated heatmaps with vertical profiles

In [ ]:
radargram_dir = os.path.join(project_root, "products", "figures", "gpr_figures", "ilaria_figures")

In [ ]:
# ── Firn cover grids for ice-temperature profile overlays ────────────────────
from affine import Affine

firn_dir = project_root + '/results/firn_grids/'

def _read_arc_grid(path):
    """Read an Arc ASCII (.grid) file → (array, Affine transform)."""
    with open(path) as f:
        ncols     = int(  f.readline().split()[1])
        nrows     = int(  f.readline().split()[1])
        xllcorner = float(f.readline().split()[1])
        yllcorner = float(f.readline().split()[1])
        cellsize  = float(f.readline().split()[1])
        nodata    = float(f.readline().split()[1])
        data = np.array(f.read().split(), dtype=np.float32).reshape(nrows, ncols)
    data[data == nodata] = np.nan
    if xllcorner < 1_000_000:   # LV03 → LV95
        xllcorner += 2_000_000
        yllcorner += 1_000_000
    tfm = Affine(cellsize, 0, xllcorner, 0, -cellsize, yllcorner + nrows * cellsize)
    return data, tfm

firn_grid_AH, firn_tfm_AH = _read_arc_grid(firn_dir + 'current/firnthick_alphubel.grid')
firn_grid_CJ, firn_tfm_CJ = _read_arc_grid(firn_dir + 'current/firnthick_felskinn.grid')
firn_grid_HS, firn_tfm_HS = _read_arc_grid(firn_dir + 'current/firnthick_hohsaas.grid')
print("Firn grids loaded — AH, CJ, HS")

firn_grid_AH_2010, firn_tfm_AH_2010 = _read_arc_grid(firn_dir + 'snapshots/firn2010_alphubel.grid')
firn_grid_CJ_2010, firn_tfm_CJ_2010 = _read_arc_grid(firn_dir + 'snapshots/firn2010_felskinn.grid')
firn_grid_HS_2010, firn_tfm_HS_2010 = _read_arc_grid(firn_dir + 'snapshots/firn2010_hohsaas.grid')

In [ ]:
output_path = os.path.join(project_root, "products", "figures", "icetemp_results") + "/"

from src.plot_composer import compose_icetemp_and_vertical_profiles
labels_AH = ['AH1G', 'AH2G', 'AH3G', 'AH3TT']
label_colors_AH = build_profile_color_map(labels_AH)

corner_marker = [("AH", "#1f77b4"), ("CJ", "#2ca02c"), ("HS", "#ff7f0e")]

fig_left, ax = gprp.plot_icetemp_profile(
    profile_df=prof_AH,
    borehole_coords_df=borehole_coordinates,
    temp_data_dict=temp_data_dict_AH,
    label_colors=label_colors_AH,
    depth_dict=depth_dict_AH,
    flip=False,          # or 'auto'
    n_depth=400,
    n_elev=600,
    temp_step=0.25,
    plot_contours=True,
    break_threshold=50.0,  # optional (default 50.0)
    smooth_sigma=0.0,      # increase (e.g. 20) to smooth surface/bed & CTS mask
    show_cts=True,
    adjust_cts_for_pressure=True,
    cts_tol=0.1,
    bed_uncertainty=5.0,       # ±2 m whiskers along bed
    bed_unc_every=40.0,        # every 40 m (halved whisker density)
    bed_unc_color= "grey",
    bh_marker_size=6.0,
    bh_line_lw=1.2,
    bed_unc_lw=1.2,
    bed_unc_capsize=4.0,
    panel_tag="L4",
    panel_tag_color="black",
    tag_bbox={"facecolor":"white","edgecolor":"black","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    cbar_min=-3.5,
    cbar_tick_step=0.5,
    rbf_smooth=0.05,
    firn_grid=firn_grid_AH,
    firn_tfm=firn_tfm_AH,
    firn_year=2025,
    firn_grid2=firn_grid_AH_2010,
    firn_tfm2=firn_tfm_AH_2010,
    firn_year2=2010,

    zaa_depth=21.7,  # from notebook 5's ZAA fit (Alphubel)
    legend_fontsize=12,  # matches panel b's base_fontsize (notebook 3)
)


# Right panel: path to vertical profiles image
right_img = os.path.join(project_root, "products", "figures", "icetemp_results", "geoprecision", "AH_profiles.png")

# set path to radargram
radargram_path = radargram_dir + "/final/AH_bed_new_version.png"

# Output
out_png = os.path.join(project_root, "figures", "paper", "fig05_AH_profile_plus_verticals.pdf")

# Compose A (left) + B (right)
fig_combo, _ = compose_icetemp_and_vertical_profiles(
    left=fig_left,
    right=right_img,
    radargram=radargram_path,
    figsize=(14.5, 14.5),
    dpi=300,
    width_ratios="auto",   # <- let the function compute from image aspect
    wspace=0.03,
    labels=("a", "b", "c"),
    label_fontsize=22,
    hspace=-0.16,
    savefig_path=out_png,
)

In [ ]:
labels_CJ = ['CJ1G', 'CJ2G', 'CJ1TT', 'CJ2TT','CJ3TT','CJ4TT']
label_colors_CJ = build_profile_color_map(labels_CJ)

fig_left, ax = gprp.plot_icetemp_profile(
    profile_df=prof_cj,
    borehole_coords_df=borehole_coordinates,
    label_colors=label_colors_CJ,
    temp_data_dict=temp_data_dict_CJ,
    depth_dict=depth_dict_CJ,
    flip=True,
    n_depth=400,
    n_elev=600,
    temp_step=0.25,
    plot_contours=True,
    break_threshold=10.0,
    smooth_sigma=50.0,
    show_cts=True,
    adjust_cts_for_pressure=True,
    cts_tol=0.1,
    bed_uncertainty=5.0,       # ±2 m whiskers along bed
    bed_unc_every=60.0,        # every 60 m (halved whisker density)
    bed_unc_color= "grey",
    bh_marker_size=6.0,
    bh_line_lw=1.2,
    bed_unc_lw=1.2,
    bed_unc_capsize=4.0,
    panel_tag="L47",
    panel_tag_color="black",
    tag_bbox={"facecolor":"white","edgecolor":"black","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    cbar_min=-3.5,
    cbar_tick_step=0.5,
    firn_offset=1.2,
    firn_grid=firn_grid_CJ,
    firn_tfm=firn_tfm_CJ,
    firn_year=2025,
    firn_grid2=firn_grid_CJ_2010,
    firn_tfm2=firn_tfm_CJ_2010,
    firn_year2=2010,

    zaa_depth=19.8,  # from notebook 5's ZAA fit (Chessjen)
    legend_fontsize=14,  # matches panel b's base_fontsize (notebook 3 default)
)


right_img = os.path.join(project_root, "products", "figures", "icetemp_results", "geoprecision", "CJ_profiles.png")
out_png = os.path.join(project_root, "figures", "paper", "fig06_CJ_profile_plus_verticals.pdf")

# set path to radargram
radargram_path = radargram_dir + "/final/CJ_bed_new_version.png"

fig_combo, _ = compose_icetemp_and_vertical_profiles(
    left=fig_left,
    right=right_img,
    radargram=radargram_path,
    hspace=-0.12,
    figsize=(14.5, 14.5),
    dpi=300,
    width_ratios="auto",
    wspace=0.03,
    labels=("a", "b", "c"),
    label_fontsize=22,
    savefig_path=out_png,
)

In [ ]:
# Hohsaas labels and colors
labels_HS = ['HS1G', 'HS2G', 'HS3G', 'HS1TT', 'HS2TT']
label_colors_HS = build_profile_color_map(labels_HS)

# Extract temperature and depth dictionaries
temp_data_dict_HS_drone_1, depth_dict_HS_drone_1 = create_temp_depth_dicts(thermistor_data_HS_drone_1, depth_data_HS_drone_1)

fig_left, ax = gprp.plot_icetemp_profile(
    profile_df=prof_hs_dronegpr1,
    borehole_coords_df=borehole_coordinates,
    temp_data_dict=temp_data_dict_HS_drone_1,
    depth_dict=depth_dict_HS_drone_1,
    label_colors = label_colors_HS,
    flip=False,          # or 'auto'
    n_depth=100,
    n_elev=200,
    temp_step=0.25,
    plot_contours=True,
    break_threshold=50.0,  # optional (default 50.0)
    smooth_sigma=6.0,      # increase (e.g. 20) to smooth surface/bed & CTS mask
    show_cts=True,
    adjust_cts_for_pressure=True,
    cts_tol=0.1,
    bed_uncertainty=5.0,       # ±2 m whiskers along bed
    bed_unc_every=20.0,        # every 20 m
    bed_unc_color= "grey",
    bh_marker_size=6.0,
    bh_line_lw=1.2,
    bed_unc_lw=1.2,
    bed_unc_capsize=4.0,
    panel_tag="L1",
    panel_tag_color="darkorange",
    tag_bbox={"facecolor":"white","edgecolor":"darkorange","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    cbar_min=-3.5,
    cbar_tick_step=0.5,
    export_txt_path=project_root + "/results/interpolated_icetemps/20250916_icetemps_HS_dronegpr1_profile.txt",
    export_borehole_txt_path=project_root + "/results/interpolated_icetemps/20250916_icetemps_HS_boreholes_dronegpr1.txt",
    continuous_cmap=False,
    hatch_regions=[(245, 400)],
    hatch_pattern='',              # Empty = no hatching
    hatch_fill_color='white',  # Solid fill
    hatch_alpha=0.8,               # 50% transparent
    firn_grid=firn_grid_HS,
    firn_tfm=firn_tfm_HS,
    firn_year=2025,
    firn_grid2=firn_grid_HS_2010,
    firn_tfm2=firn_tfm_HS_2010,
    firn_year2=2010,
)

# Add corner marker
fig_left.text(
    0.04, 0.13, "HS",
    transform=fig_left.transFigure,
    ha="left", va="bottom",
    fontsize=22, fontweight="bold",
    color="k",
    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="k", alpha=0.7),
    zorder=40,
    clip_on=True,
)

right_img = os.path.join(project_root, "products", "figures", "icetemp_results", "geoprecision", "HS_profiles_drone_1.png")
out_png = output_path + "interpolated_figures/HS_dronegpr1_plus_verticals.pdf"

# set path to radargram
radargram_path = radargram_dir + "/final/HS_centerline1_new_version.png"

fig_combo, _ = compose_icetemp_and_vertical_profiles(
    left=fig_left,
    right=right_img,
    radargram=radargram_path,
    figsize=(14.5, 14.5),
    hspace=-0.17,
    dpi=300,
    width_ratios="auto",
    wspace=0.0,
    labels=("a", "b", "c"),
    label_fontsize=22,
    savefig_path=out_png,
)

In [ ]:
# Hohsaas labels and colors
labels_HS = ['HS1G', 'HS2G', 'HS3G', 'HS1TT', 'HS2TT']
label_colors_HS = build_profile_color_map(labels_HS)

# Extract temperature and depth dictionaries
temp_data_dict_HS_drone_2, depth_dict_HS_drone_2 = create_temp_depth_dicts(thermistor_data_HS_drone_2, depth_data_HS_drone_2)

fig_left, ax = gprp.plot_icetemp_profile(
    profile_df=prof_hs_dronegpr2,
    borehole_coords_df=borehole_coordinates,
    temp_data_dict=temp_data_dict_HS_drone_2,
    depth_dict=depth_dict_HS_drone_2,
    label_colors = label_colors_HS,
    flip=False,          # or 'auto'
    n_depth=100,
    n_elev=200,
    temp_step=0.25,
    plot_contours=True,
    break_threshold=50.0,  # optional (default 50.0)
    smooth_sigma=6.0,      # increase (e.g. 20) to smooth surface/bed & CTS mask
    show_cts=True,
    adjust_cts_for_pressure=True,
    cts_tol=0.05,
    bed_uncertainty=5.0,       # ±2 m whiskers along bed
    bed_unc_every=20.0,        # every 20 m
    bed_unc_color= "grey",
    bh_marker_size=6.0,
    bh_line_lw=1.2,
    bed_unc_lw=1.2,
    bed_unc_capsize=4.0,
    panel_tag="L2",
    panel_tag_color="darkorange",
    tag_bbox={"facecolor":"white","edgecolor":"darkorange","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    cbar_min=-3.5,
    cbar_tick_step=0.5,
    export_txt_path=project_root + "/results/interpolated_icetemps/20250916_icetemps_HS_dronegpr2_profile.txt",
    export_borehole_txt_path=project_root + "/results/interpolated_icetemps/20250916_icetemps_HS_boreholes_dronegpr2.txt",
    continuous_cmap=False,
    rbf_smooth=0.12,
    hatch_regions=[(260, 400)],
    hatch_pattern='',              # Empty = no hatching
    hatch_fill_color='white',  # Solid fill
    hatch_alpha=0.9,               # 50% transparent
    firn_grid=firn_grid_HS,
    firn_tfm=firn_tfm_HS,
    firn_year=2025,
    firn_grid2=firn_grid_HS_2010,
    firn_tfm2=firn_tfm_HS_2010,
    firn_year2=2010,
)

# Add corner marker
fig_left.text(
    0.04, 0.13, "HS",
    transform=fig_left.transFigure,
    ha="left", va="bottom",
    fontsize=22, fontweight="bold",
    color="k",
    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="k", alpha=0.7),
    zorder=40,
    clip_on=True,
)

right_img = os.path.join(project_root, "products", "figures", "icetemp_results", "geoprecision", "HS_profiles_drone_2.png")
out_png = output_path + "interpolated_figures/HS_dronegpr2_plus_verticals.pdf"

# set path to radargram
radargram_path = radargram_dir + "/final/HS_centerline2_new_version.png"

fig_combo, _ = compose_icetemp_and_vertical_profiles(
    left=fig_left,
    right=right_img,
    radargram=radargram_path,
    figsize=(14.5, 14.5),
    hspace=-0.17,
    dpi=300,
    width_ratios="auto",
    wspace=0.0,
    labels=("a", "b", "c"),
    label_fontsize=22,
    savefig_path=out_png,
)

In [ ]:
# --- Combined HS figure: [L1 | L2 | depth profiles] + [L2 radargram] ---

labels_HS       = ['HS1G', 'HS2G', 'HS3G', 'HS1TT', 'HS2TT']
label_colors_HS = build_profile_color_map(labels_HS)
_FONTSIZE = 12

# ── Load radargram image upfront to compute correct height ────────────────────
_rad_path = radargram_dir + "/final/HS_centerline2_new_version.png"
radargram_img = mpimg.imread(_rad_path)
_img_h, _img_w = radargram_img.shape[:2]

# ── Pre-size the bottom row so the axes box matches image aspect ratio ─────────
_FIG_W      = 14.8
_TOP_H_IN   = 4.0
_RAD_W_IN   = _FIG_W * 0.91
_RAD_H_IN   = _RAD_W_IN * (_img_h / _img_w)
_FIG_H      = _TOP_H_IN + _RAD_H_IN + 2.6
_hr1        = 1.55 * (_RAD_H_IN / _TOP_H_IN)

# ── Figure layout ─────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(_FIG_W, _FIG_H), dpi=300)
# Outer GridSpec: (L1+L2 block) | depth profile  — controls gap between them
gs = gridspec.GridSpec(
    2, 2, figure=fig,
    height_ratios=[1.55, _hr1],
    width_ratios=[2.2, 0.45],
    hspace=0.13,
    wspace=0.12,   # gap between L1/L2 block and depth profile (adjustable)
)
# Inner GridSpec inside the L1+L2 block — controls gap between the two profiles
gs_ab = gridspec.GridSpecFromSubplotSpec(
    1, 2, subplot_spec=gs[0, 0],
    width_ratios=[1, 1],
    wspace=0.06,   # gap between L1 and L2 (fixed)
)

ax_L1  = fig.add_subplot(gs_ab[0, 0])
ax_L2  = fig.add_subplot(gs_ab[0, 1], sharey=ax_L1)
ax_dep = fig.add_subplot(gs[0, 1])
ax_rad = fig.add_subplot(gs[1, 0:2])

# ── Shared profile kwargs ─────────────────────────────────────────────────────
_prof_kw = dict(
    borehole_coords_df=borehole_coordinates,
    label_colors=label_colors_HS,
    flip=False,
    n_depth=100, n_elev=200, temp_step=0.25,
    plot_contours=True, break_threshold=50.0, smooth_sigma=6.0,
    show_cts=True, adjust_cts_for_pressure=True, cts_tol=0.1,
    bed_uncertainty=5.0, bed_unc_every=40.0, bed_unc_color='grey',  # halved whisker density
    bh_marker_size=6.0, bh_line_lw=1.2, bed_unc_lw=1.2, bed_unc_capsize=4.0,
    panel_tag_color='darkorange',
    tag_bbox={"facecolor":"white","edgecolor":"darkorange",
              "boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    tag_loc='BR',
    cbar_min=-3.5, cbar_tick_step=0.5,
    show_cbar=False, return_mappable=True,
    continuous_cmap=False,
    hatch_pattern='', hatch_fill_color='white', hatch_alpha=0.8,
    firn_grid=firn_grid_HS, firn_tfm=firn_tfm_HS, firn_year=2025,
    firn_grid2=firn_grid_HS_2010, firn_tfm2=firn_tfm_HS_2010, firn_year2=2010,
    borehole_clip_buffer=20,
    base_fontsize=_FONTSIZE,
    zaa_depth=18.4,  # from notebook 5's ZAA fit (Hohsaas)
    legend_fontsize=_FONTSIZE - 2,  # matches panel b's (ax_dep) legend fontsize
)

# ── L1 cross-section ──────────────────────────────────────────────────────────
_, _, im, norm, levels = gprp.plot_icetemp_profile(
    profile_df=prof_hs_dronegpr1,
    temp_data_dict=temp_data_dict_HS_drone_1,
    depth_dict=depth_dict_HS_drone_1,
    ax=ax_L1,
    panel_tag='L1',
    hatch_regions=[(245, 400)],
    rbf_smooth=0.05,
    **_prof_kw,
)
ax_L1.xaxis.set_major_locator(ticker.MultipleLocator(50))
ax_L1.tick_params(labelsize=_FONTSIZE - 1)

# ── L2 cross-section ──────────────────────────────────────────────────────────
_, _, im_L2, norm_L2, levels_L2 = gprp.plot_icetemp_profile(
    profile_df=prof_hs_dronegpr2,
    temp_data_dict=temp_data_dict_HS_drone_2,
    depth_dict=depth_dict_HS_drone_2,
    ax=ax_L2,
    panel_tag='L2',
    hatch_regions=[(260, 400)],
    rbf_smooth=0.12,
    **_prof_kw,
)
ax_L2.set_ylabel('')
plt.setp(ax_L2.get_yticklabels(), visible=False)
ax_L2.xaxis.set_major_locator(ticker.MultipleLocator(50))
ax_L2.tick_params(labelsize=_FONTSIZE - 1)
_leg = ax_L2.get_legend()
if _leg:
    _leg.remove()

# ── Shared colorbar below L1 + L2 ────────────────────────────────────────────
sm = ScalarMappable(norm=norm_L2, cmap=im_L2.cmap if im_L2 is not None else plt.cm.Blues)
sm.set_array([])
lo_edge = float(levels_L2.min())
ticks_asc = np.arange(lo_edge, 0.0 + 1e-12, 0.5)
cb = fig.colorbar(sm, ax=[ax_L1, ax_L2], location='bottom', orientation='horizontal',
                  fraction=0.06, pad=0.17, aspect=50, anchor=(0.5, 0.0), extend='max')
cb.set_ticks(ticks_asc)
cb.set_ticklabels([f"{t:.1f}" for t in ticks_asc])
cb.set_label('Ice Temperature [°C]', fontsize=_FONTSIZE)
cb.ax.tick_params(labelsize=_FONTSIZE - 1)

# ── Combined depth profiles ───────────────────────────────────────────────────
def _extract_depths(d):
    if isinstance(d, dict):
        return np.array(list(d.values()), dtype=float)
    return np.atleast_1d(d).astype(float)

def _extract_temps(t):
    import pandas as pd
    if isinstance(t, pd.DataFrame):
        return t.mean(axis=0).values.astype(float)
    if isinstance(t, pd.Series):
        return t.values.astype(float)
    arr = np.atleast_1d(t)
    return np.nanmean(arr, axis=0).astype(float) if arr.ndim > 1 else arr.astype(float)

all_temp  = {**temp_data_dict_HS_drone_2, **temp_data_dict_HS_drone_1}
all_depth = {**depth_dict_HS_drone_2,     **depth_dict_HS_drone_1}

for label in ['HS1G', 'HS2G', 'HS3G', 'HS1TT', 'HS2TT']:
    if label not in all_temp:
        continue
    temps  = _extract_temps(all_temp[label])
    depths = _extract_depths(all_depth[label])
    if len(temps) != len(depths):
        print(f"Warning: {label} shape mismatch ({len(temps)} vs {len(depths)}) — skipping")
        continue
    temps = np.where(temps > 0.05, np.nan, temps)
    ax_dep.plot(temps, depths, '-o', color=label_colors_HS[label],
                lw=1.8, ms=5, label=label)

ax_dep.invert_yaxis()
ax_dep.set_xlim(-1.75, 0.1)
ax_dep.axvspan(0, 0.1, color='lightgrey', alpha=0.4, zorder=0)
ax_dep.axhline(18.4, color='black', linestyle='-', linewidth=1.2, zorder=1)  # matches panel a's ZAA fit (Hohsaas)
ax_dep.text(0.02, 18.4, 'ZAA', transform=ax_dep.get_yaxis_transform(), ha='left', va='bottom', fontsize=_FONTSIZE - 2, color='black', zorder=2)
ax_dep.set_xlabel('Ice Temperature [°C]', fontsize=_FONTSIZE)
ax_dep.set_ylabel('Depth [m]', fontsize=_FONTSIZE)
ax_dep.legend(fontsize=_FONTSIZE - 2, loc='lower left', frameon=True,
              fancybox=False, edgecolor='black')
ax_dep.grid(True, alpha=0.3)
ax_dep.tick_params(labelsize=_FONTSIZE - 1)

# ── L2 radargram ──────────────────────────────────────────────────────────────
ax_rad.imshow(radargram_img, aspect='auto')
ax_rad.axis('off')
# L2 tag: same visual size as L1/L2 tags inside the profile panels
ax_rad.text(0.97, 0.94, 'L2',
            transform=ax_rad.transAxes, fontsize=_FONTSIZE + 4, fontweight='bold',
            color='darkorange', ha='right', va='top', zorder=10,
            bbox={"facecolor":"white","edgecolor":"darkorange",
                  "boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75})

# ── Panel labels ──────────────────────────────────────────────────────────────
LABEL_KW = dict(fontsize=_FONTSIZE + 3, fontweight='bold',
                va='top', ha='right',
                bbox=dict(facecolor='none', edgecolor='none', pad=1.0),
                clip_on=False)
ax_L1.text(-0.04, 1.0, '(a)', transform=ax_L1.transAxes, **LABEL_KW, zorder=20)
ax_dep.text(-0.09, 1.0, '(b)', transform=ax_dep.transAxes, **LABEL_KW, zorder=20)
ax_rad.text(0.06, 0.95, '(c)', transform=ax_rad.transAxes, **LABEL_KW, zorder=20)

# ── Fix radargram axes to exact correct aspect ratio (post-layout) ────────────
fig.canvas.draw()
_renderer = fig.canvas.get_renderer()
_fig_bbox  = fig.bbox

# Use tight bboxes so the radargram spans the full visual width of the top row
# (i.e. including the y-axis labels of ax_L1 on the left)
_tb_L1  = ax_L1.get_tightbbox(_renderer)
_tb_dep = ax_dep.get_tightbbox(_renderer)
rad_x0  = _tb_L1.x0  / _fig_bbox.width - 0.023  # shift left to align with panel (a)
rad_x1  = _tb_dep.x1 / _fig_bbox.width
rad_w   = rad_x1 - rad_x0  # right edge unchanged
rad_w_in = rad_w * fig.get_figwidth()
rad_h_in = rad_w_in * (_img_h / _img_w)
rad_h   = rad_h_in / fig.get_figheight()

pos_rad = ax_rad.get_position()
ax_rad.set_position([rad_x0, pos_rad.y0, rad_w, rad_h])

# ── Save ──────────────────────────────────────────────────────────────────────
out_combined = os.path.join(project_root, "figures", "paper", "fig07_HS_combined_L1_L2.pdf")
plt.savefig(out_combined, dpi=300, bbox_inches='tight')
plt.savefig(out_combined.replace('.pdf', '.png'), dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved: {out_combined}')


In [ ]:
# Colors per borehole label for each panel
labels_HS1 = ['HS1G','HS2TT']     # Line 39
labels_HS2 = ['HS2G','HS1TT']     # Line 40
label_colors_HS1 = build_profile_color_map(labels_HS1)
label_colors_HS2 = build_profile_color_map(labels_HS2)

# Left: side-by-side Hohsaas interpolated temperature profiles
fig_left, axs = gprp.plot_icetemp_profiles_side_by_side(
    profiles=[
        (prof_hs2, temp_data_dict_HS2, depth_dict_HS2),  # L40
        (prof_hs1, temp_data_dict_HS1, depth_dict_HS1),  # L39
    ],
    borehole_coords_df=borehole_coordinates,
    panel_tags=["L40","L39"],
    flips=[False, False],
    n_depth=400, n_elev=600, temp_step=0.25,
    plot_contours=True, break_threshold=10.0, smooth_sigma=50.0,
    show_cts=True, adjust_cts_for_pressure=True, cts_tol=0.05,
    borehole_clip_buffer=[10.0, 10.0],
    label_colors_list=[label_colors_HS2, label_colors_HS1],
    figsize=(14.5, 7.5),
    panel_gap=0.16,
    bh_label_offset=3.0,
    y_top_margin=20.0,
    keep_true_slope=True,
    show_all_y_ticks=True,
    bed_uncertainty=[5.0, 5.0],
    bed_unc_every=20.0,
    bed_unc_color="#E69F00",
    tag_bbox={"facecolor":"white","edgecolor":"black","boxstyle":"round,pad=0.25","linewidth":1.2,"alpha":0.75},
    cbar_min=-3.5,
    cbar_tick_step=0.5,   # shared colorbar ticks
    bh_marker_size=10.0,
    bh_line_lw=2.0,
    bed_unc_lw=2.0,
    bed_unc_capsize=4.0,

    zaa_depths=[18.4, 18.4],  # from notebook 5's ZAA fit (Hohsaas)
)

# After fig_left, before compose_icetemp_and_vertical_profiles
fig_left.text(
    0.07, 0.10, "HS",
    transform=fig_left.transFigure,
    ha="left", va="bottom",
    fontsize=29, fontweight="bold",
    color="k",
    bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="k", alpha=0.7),
    zorder=40,
    clip_on=True,
)

# Right: path to vertical profiles image for Hohsaas
right_img = os.path.join(project_root, "products", "figures", "icetemp_results", "geoprecision", "HS_profiles.png")

# Output path
out_png = output_path + "interpolated_figures/HS_side_by_side_plus_verticals.pdf"

# Compose combo figure (A: interpolated, B: vertical profiles)
from src.plot_composer import compose_icetemp_and_vertical_profiles
fig_combo, _ = compose_icetemp_and_vertical_profiles(
    left=fig_left,
    right=right_img,
    figsize=(14.5, 6.0),
    dpi=300,
    width_ratios="auto",
    wspace=0.0,
    labels=("a", "b"),
    label_fontsize=22,
    savefig_path=out_png,
)

### Compress final figures

In [ ]:
# Compress heatmap + radargram figures to reduce file size
hs_combined = os.path.join(project_root, "figures", "paper", "fig07_HS_combined_L1_L2.pdf")
alphubel_dir   = os.path.join(project_root, "figures", "paper", "fig05_AH_profile_plus_verticals.pdf")
chessjen_dir   = os.path.join(project_root, "figures", "paper", "fig06_CJ_profile_plus_verticals.pdf")

# compress figures -- ladder steps DPI down within /prepress (best JPEG
# quality) with a dedicated rung just under each source's native-resolution
# cliff (~200dpi for these radargrams), so each figure lands as close to full
# native quality as the 2 MB budget allows.
compress_figure_inplace(hs_combined, max_size_mb=2.0)
compress_figure_inplace(alphubel_dir, max_size_mb=2.0)
compress_figure_inplace(chessjen_dir, max_size_mb=2.0)

### Compress Gabrielas figures

In [ ]:
amplitude_radargrams = os.path.join(project_root, "products", "figures", "gpr_figures", "gabriela_figures", "paper_three_boreholes_crossline.png")
crossline_profiles_HS = os.path.join(project_root, "products", "figures", "gpr_figures", "gabriela_figures", "paper_radargrams_inline (1).png")

# compress figures
compress_figure_inplace(amplitude_radargrams, max_size_mb=2.0)
compress_figure_inplace(crossline_profiles_HS, max_size_mb=2.0)

## Empirical justification for RBF vertical coordinate scaling

Computes mean vertical and horizontal temperature gradients from the borehole data to empirically justify the scaling factor `w = 2.5` used in the RBF interpolation. The ratio of vertical to horizontal gradients provides a data-driven basis for the choice of `w`.

In [ ]:
bh_coords = pd.read_csv(
    os.path.join(project_root, "data", "borehole_settings", "thermistor_coordinates.csv")
)
bh_coords = bh_coords.set_index('name')


def vertical_gradients(temp_dict, depth_dict):
    """
    Mean absolute vertical temperature gradient within each borehole (deg C / m).
    Only computed where at least 2 sensors exist per borehole.
    """
    grads = []
    for bh, temps in temp_dict.items():
        depths = depth_dict.get(bh, {})
        pairs = [
            (depths[s], float(temps[s]))
            for s in temps.index
            if s in depths and depths[s] is not None and not np.isnan(float(temps[s]))
        ]
        pairs = sorted(pairs, key=lambda x: x[0])
        if len(pairs) < 2:
            continue
        for i in range(1, len(pairs)):
            dz = pairs[i][0] - pairs[i - 1][0]
            dT = abs(pairs[i][1] - pairs[i - 1][1])
            if dz > 0:
                grads.append(dT / dz)
    return np.array(grads)


def horizontal_gradients(temp_dict, depth_dict, bh_coords_df, depth_tolerance=3.0):
    """
    Mean absolute horizontal temperature gradient between boreholes
    at comparable depths (within depth_tolerance metres of each other).
    """
    grads = []
    bh_names = list(temp_dict.keys())
    for i in range(len(bh_names)):
        for j in range(i + 1, len(bh_names)):
            bh_i, bh_j = bh_names[i], bh_names[j]
            if bh_i not in bh_coords_df.index or bh_j not in bh_coords_df.index:
                continue
            dx = float(bh_coords_df.loc[bh_i, 'x'] - bh_coords_df.loc[bh_j, 'x'])
            dy = float(bh_coords_df.loc[bh_i, 'y'] - bh_coords_df.loc[bh_j, 'y'])
            horiz_dist = np.hypot(dx, dy)
            if horiz_dist == 0:
                continue
            depths_i = depth_dict.get(bh_i, {})
            depths_j = depth_dict.get(bh_j, {})
            temps_i = temp_dict[bh_i]
            temps_j = temp_dict[bh_j]
            for s_i in temps_i.index:
                if s_i not in depths_i or depths_i[s_i] is None:
                    continue
                for s_j in temps_j.index:
                    if s_j not in depths_j or depths_j[s_j] is None:
                        continue
                    if abs(depths_i[s_i] - depths_j[s_j]) <= depth_tolerance:
                        dT = abs(float(temps_i[s_i]) - float(temps_j[s_j]))
                        grads.append(dT / horiz_dist)
    return np.array(grads)


# For HS, merge the two drone profile dicts to use all 5 boreholes together
temp_data_dict_HS_all = {**temp_data_dict_HS_drone_1, **temp_data_dict_HS_drone_2}
depth_dict_HS_all     = {**depth_dict_HS_drone_1,     **depth_dict_HS_drone_2}

datasets = {
    'AH': (temp_data_dict_AH, depth_dict_AH),
    'CJ': (temp_data_dict_CJ, depth_dict_CJ),
    'HS': (temp_data_dict_HS_all, depth_dict_HS_all),
}

rows = []
for abbr, (temp_dict, depth_dict) in datasets.items():
    v_grads = vertical_gradients(temp_dict, depth_dict)
    h_grads = horizontal_gradients(temp_dict, depth_dict, bh_coords)
    mean_v  = float(np.mean(v_grads))  if len(v_grads) else np.nan
    mean_h  = float(np.mean(h_grads))  if len(h_grads) else np.nan
    ratio   = mean_v / mean_h          if mean_h > 0   else np.nan
    rows.append(dict(
        Glacier    = abbr,
        n_vert     = len(v_grads),
        mean_vert  = round(mean_v,  4),
        n_horiz    = len(h_grads),
        mean_horiz = round(mean_h,  4),
        ratio_v_h  = round(ratio,   2),
    ))
    print(f"{abbr}: vertical {mean_v:.4f} deg C/m (n={len(v_grads)}),  "
          f"horizontal {mean_h:.4f} deg C/m (n={len(h_grads)}),  "
          f"ratio = {ratio:.2f}")

summary = pd.DataFrame(rows).set_index('Glacier')
print()
print(summary.to_string())
print(f"\nOverall mean ratio (all glaciers): {summary['ratio_v_h'].mean():.2f}")
print(f"=> Suggested scaling factor w ~ {summary['ratio_v_h'].mean():.1f}")

---

## Part 2 — Zero Annual Amplitude and Thermistor Metrics

In [ ]:
# set main icetemp data dir
TT_icetemp_dir = os.path.join(ICETEMP_ROOT, "NTC_tynitag", "temperature_data", "full_timeseries") + "/"
TT_settings_dir = os.path.join(project_root, "data", "borehole_settings") + "/"

# set output dir for figures
output_dir = os.path.join(project_root, "products", "figures", "icetemp_results", "geoprecision")

# set tiny tag data dir
HS1TT_dir = TT_icetemp_dir + "HS1TT_20240808_20250927_spliced.csv" # HS1TT
HS2TT_dir = TT_icetemp_dir + "HS2TT_20240808_20250927_spliced.csv" # HS2TT
AH1TT_dir = TT_icetemp_dir + "AH1TT_20250322_20250916.csv" #AH1TT
AH2TT_dir = TT_icetemp_dir + "AH2TT_20240821_20250916_spliced.csv" #AH2TT
CJ1TT_dir = TT_icetemp_dir + "CJ1TT_20240809_20250808_spliced.csv" #CJ1TT
CJ2TT_dir = TT_icetemp_dir + "CJ2TT_20240809_20250808_spliced.csv" #CJ2TT
SR1TT_dir = TT_icetemp_dir + "SR1TT_20240806_20250724_spliced.csv" #SR1TT
SR2TT_dir = TT_icetemp_dir + "SR2TT_20240806_20250724_spliced.csv" #SR2TT
GT1TT_dir = TT_icetemp_dir + "GT1TT_20240807_20250723_spliced.csv" #GT1TT
GT2TT_dir = TT_icetemp_dir + "GT2TT_20240807_20250723_spliced.csv" #GT2TT
CT1TT_dir = TT_icetemp_dir + "CT1TT_20240828_20250905.csv" #CT1TT
CT2TT_dir = TT_icetemp_dir + "CT2TT_20240828_20250905_spliced.csv" #CT2TT

## set path to current depth file

# Chessjen depth files
depth_CJ1TT = TT_settings_dir + "thermistor_settings_cj1tt.csv"
depth_CJ2TT = TT_settings_dir + "thermistor_settings_cj2tt.csv"

# Alphubel depth files
depth_AH1TT = TT_settings_dir + "thermistor_settings_ah1tt.csv"
depth_AH2TT = TT_settings_dir + "thermistor_settings_ah2tt.csv"

# Hohsaas depth files
depth_HS1TT = TT_settings_dir + "thermistor_settings_hs1tt.csv"
depth_HS2TT = TT_settings_dir + "thermistor_settings_hs2tt.csv"

# Sex Rouges depth files
depth_SR1TT = TT_settings_dir + "thermistor_settings_sr1tt.csv"
depth_SR2TT = TT_settings_dir + "thermistor_settings_sr2tt.csv"

# Glacier de Tortin depth files
depth_GT1TT = TT_settings_dir + "thermistor_settings_gt1tt.csv"
depth_GT2TT = TT_settings_dir + "thermistor_settings_gt2tt.csv"

# Corvatsch depth files
depth_CT1TT = TT_settings_dir + "thermistor_settings_ct1tt.csv"
depth_CT2TT = TT_settings_dir + "thermistor_settings_ct2tt.csv"

## generate a thermistor data object

# Chessjen boreholes
CJ1TT = ThermistorData(CJ1TT_dir, ",", depth_CJ1TT)
CJ2TT = ThermistorData(CJ2TT_dir, ",", depth_CJ2TT)

# Alphubel boreholes
AH1TT = ThermistorData(AH1TT_dir, ",", depth_AH1TT)
AH2TT = ThermistorData(AH2TT_dir, ",", depth_AH2TT)

# Hohsaas boreholes (HS)
HS1TT = ThermistorData(HS1TT_dir, ",", depth_HS1TT)
HS2TT = ThermistorData(HS2TT_dir, ",", depth_HS2TT)

# Sex Rouges boreholes (SR)
SR1TT = ThermistorData(SR1TT_dir, ",", depth_SR1TT)
SR2TT = ThermistorData(SR2TT_dir, ",", depth_SR2TT)

# Glacier de Tortin boreholes (GT)
GT1TT = ThermistorData(GT1TT_dir, ",", depth_GT1TT)
GT2TT = ThermistorData(GT2TT_dir, ",", depth_GT2TT)

# Corvatsch boreholes (CT)
CT1TT = ThermistorData(CT1TT_dir, ",", depth_CT1TT)
CT2TT = ThermistorData(CT2TT_dir, ",", depth_CT2TT)

In [ ]:
# TinyTag offsets (0°C offsets)
offsets_TT_path = os.path.join(ICETEMP_ROOT, "NTC_tynitag", "calibration_data", "all_logger_offsets.csv")
offsets_TT = pd.read_csv(offsets_TT_path)

# Define TinyTag entries (adjust logger_id if needed)
tt_entries = [
    {"name": "HS1TT", "glacier": "Hohsaas", "thermistor": HS1TT, "logger_id": "5",  "depth_file": depth_HS1TT},
    {"name": "HS2TT", "glacier": "Hohsaas", "thermistor": HS2TT, "logger_id": "6",  "depth_file": depth_HS2TT},
    {"name": "AH1TT", "glacier": "Alphubel","thermistor": AH1TT, "logger_id": "9",  "depth_file": depth_AH1TT},
    {"name": "AH2TT", "glacier": "Alphubel","thermistor": AH2TT, "logger_id": "10",  "depth_file": depth_AH2TT},
    {"name": "CJ1TT", "glacier": "Chessjen","thermistor": CJ1TT, "logger_id": "7",  "depth_file": depth_CJ1TT},
    {"name": "CJ2TT", "glacier": "Chessjen","thermistor": CJ2TT, "logger_id": "8",  "depth_file": depth_CJ2TT},
    {"name": "SR1TT", "glacier": "SexRouges","thermistor": SR1TT, "logger_id": "1",  "depth_file": depth_SR1TT},
    {"name": "SR2TT", "glacier": "SexRouges","thermistor": SR2TT, "logger_id": "2",  "depth_file": depth_SR2TT},
    {"name": "GT1TT", "glacier": "Tortin","thermistor": GT1TT, "logger_id": "3",  "depth_file": depth_GT1TT},
    # {"name": "GT2TT", "glacier": "Tortin","thermistor": GT2TT, "logger_id": "4",  "depth_file": depth_GT2TT}, # thermistors not working
    {"name": "CT1TT", "glacier": "Corvatsch","thermistor": CT1TT, "logger_id": "11",  "depth_file": depth_CT1TT},
    # {"name": "CT2TT", "glacier": "Corvatsch","thermistor": CT2TT, "logger_id": "12",  "depth_file": depth_CT2TT}, # only one thermistor working
]
# TODO: Fill the "?" logger IDs.

# Window covering the hydrological year
win = dict(start_time="01.09.2024 00:00:00", end_time="30.09.2025 00:00:00")

# Compute ZAA per TinyTag borehole
metrics_zaa = compute_tynitag_zaa_batch(tt_entries, offsets_TT, zaa_threshold=0.1, zaa_extrapolate=True, **win)

# Aggregate by glacier for the summary plot
summary_zaa = summarize_zaa_by_glacier(metrics_zaa)
summary_zaa

In [ ]:
# Plot summary of ZAA by glacier
df = summary_zaa.sort_values("glacier").reset_index(drop=True)
glaciers = df["glacier"].tolist()
x = np.arange(len(glaciers))
y = df["zaa_mean"].to_numpy()
yerr = 0.5 * df["zaa_range"].fillna(0).to_numpy()

fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=150)
ax.errorbar(
    x, y, yerr=yerr,
    fmt="o", color="k",
    ecolor="0.35", elinewidth=1.2, capsize=4,
    label="Mean ± range/2"
)
ax.set_ylabel("ZAA depth [m]")
ax.set_xticks(x)
ax.set_xticklabels(glaciers, rotation=0)

format_plot(ax=ax, title="TinyTag ZAA depth by glacier",
            legend_loc="upper right", x_tick_rotation=0,
            adjust_linewidths=False)
plt.show()